# Manual Gegenpressing Annotation

Notebook ini membuat ground truth perilaku gegenpressing secara terpisah dari outcome quick regain. Full queue berisi 264 transisi strict-valid. Tahap pertama menggunakan pilot 28 sampel: semua keputusan yang sudah tersimpan dipertahankan dan sisanya dipilih dari pertandingan lain agar tersebar lintas match serta seimbang antara quick regain dan non-regain. Outcome disembunyikan dari panel keputusan untuk mengurangi bias anotator.

Metodologi turnover, window struktural, dan kriteria strict-valid dari notebook 05 tidak diubah.

## Definisi operasional yang digunakan

Unit keputusan adalah dua detik pertama setelah kehilangan penguasaan (`T0` sampai `T0+2s`). Literatur mendefinisikan counterpressing sebagai tekanan segera setelah kehilangan bola terhadap pembawa bola atau lawan di sekitar bola. Regain dalam lima detik adalah ukuran **keberhasilan**, bukan definisi keberadaan gegenpressing.

Beri `is_gegenpressing = 1` jika terlihat reaksi tekanan segera dan disengaja, yaitu sedikitnya satu pemain losing team menutup pembawa bola/bola, atau beberapa pemain menutup ruang dan opsi umpan di sekitar bola. Tekanan harus tampak berkelanjutan, bukan hanya kebetulan berada dekat bola.

Beri `is_gegenpressing = 0` jika respons dominan adalah mundur/reorganisasi, tidak ada pemain yang segera menekan, atau kedekatan pemain tidak disertai gerak menutup lawan/ruang.

Beri `-1` (ragu) untuk bola udara/clearance, pembawa bola tidak dapat diidentifikasi, transisi terlalu singkat, atau bukti visual tidak cukup. Sampel ragu tidak dipakai sebagai kelas training sampai ditinjau ulang.

Panel menampilkan cue, bukan label otomatis:
- tekanan langsung: jarak ke bola <= 4,572 m (lima yard);
- approaching pressure: jarak <= 10 m dan jarak berkurang >= 1 m/s;
- durasi cue minimum: 0,4 detik;
- jumlah pemain dalam radius 10 m/15 m dan compactness lokal;
- lingkaran hijau menandai pemain yang memenuhi cue, sedangkan lingkaran gelap menandai proxy pembawa bola (lawan terdekat ke bola).

Ambang selain lima yard adalah parameter audit awal dan harus dievaluasi melalui agreement/analisis sensitivitas, bukan diperlakukan sebagai kebenaran taktis. Dasar metodologi: [Bauer & Anzer (2021)](https://link.springer.com/article/10.1007/s10618-021-00763-7).

In [1]:
from functools import lru_cache
from pathlib import Path
import sys

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.gegenpressing_annotation import (
    AnnotationConfig,
    annotation_progress,
    compute_annotation_context,
    ensure_pilot_selection,
    initialize_annotation_manifest,
    plot_annotation_context,
    record_annotation,
)
from src.idsse_episode_dataset import load_match

DATA_DIR = PROJECT_ROOT / 'data' / 'sportec_idsse'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EPISODE_MANIFEST_PATH = PROCESSED_DIR / 'episode_manifest.csv'
ANNOTATION_PATH = PROCESSED_DIR / 'gegenpressing_annotation_manifest.csv'
CONFIG = AnnotationConfig()
ANNOTATOR = 'rizal'


In [2]:
annotations = initialize_annotation_manifest(
    EPISODE_MANIFEST_PATH,
    ANNOTATION_PATH,
    samples_per_match=40,
    random_state=42,
    min_spacing_seconds=2.0,
)
annotations = ensure_pilot_selection(
    annotations,
    ANNOTATION_PATH,
    target_total=28,
    random_state=42,
)
pilot_queue = (
    annotations[annotations['pilot_selected']]
    .sort_values('pilot_order')
)

queue_check = (
    annotations.groupby(['match_id', 'quick_regain_success'])
    .size()
    .unstack(fill_value=0)
)
display(queue_check)
print('Pilot distribution by match:')
display(pilot_queue.groupby('match_id').size().rename('count').to_frame())
print('Pilot outcome balance (hidden during decisions):')
display(pilot_queue['quick_regain_success'].value_counts().sort_index().rename('count').to_frame())
print('Pilot progress:')
display(annotation_progress(pilot_queue).to_frame('count'))
print('Full queue progress:')
display(annotation_progress(annotations).to_frame('count'))
print('Annotation file:', ANNOTATION_PATH)

quick_regain_success,0,1
match_id,,
J03WMX,20,20
J03WN1,12,12
J03WOH,20,20
J03WOY,20,20
J03WPY,20,20
J03WQQ,20,20
J03WR9,20,20


Pilot distribution by match:


,count
match_id,
J03WMX,10
J03WN1,3
J03WOH,3
J03WOY,3
J03WPY,3
J03WQQ,3
J03WR9,3


Pilot outcome balance (hidden during decisions):


,count
quick_regain_success,
0,14
1,14


Pilot progress:


,count
total_queue,28
completed,28
remaining,0
gegenpressing,21
non_gegenpressing,4
uncertain,3


Full queue progress:


,count
total_queue,264
completed,28
remaining,236
gegenpressing,21
non_gegenpressing,4
uncertain,3


Annotation file: c:\VAEP-Track\data\processed\gegenpressing_annotation_manifest.csv


## Panel anotasi

Tentukan label dari pola gerak dan cue yang terlihat. Jangan membuka kolom `quick_regain_success` untuk memutuskan label. Confidence: `1` rendah, `2` sedang, `3` tinggi. Setiap klik langsung menyimpan CSV sehingga progres tidak hilang.

In [ ]:
@lru_cache(maxsize=1)
def load_cached_match(match_id):
    return load_match(DATA_DIR, match_id)

start_position = next((
    position + 1
    for position, value in enumerate(pilot_queue['is_gegenpressing'])
    if pd.isna(value)
), 1)
candidate_selector = widgets.IntSlider(
    value=start_position, min=1, max=len(pilot_queue), step=1,
    description='Pilot', continuous_update=False,
    layout=widgets.Layout(width='70%'),
)
confidence = widgets.ToggleButtons(
    options=[('Rendah', 1), ('Sedang', 2), ('Tinggi', 3)],
    value=2, description='Confidence',
)
notes = widgets.Textarea(
    description='Catatan',
    placeholder='Alasan keputusan atau penyebab ragu',
    layout=widgets.Layout(width='85%', height='70px'),
)
positive_button = widgets.Button(description='Gegenpressing', button_style='success')
negative_button = widgets.Button(description='Bukan', button_style='danger')
uncertain_button = widgets.Button(description='Ragu', button_style='warning')
previous_button = widgets.Button(description='Sebelumnya')
next_button = widgets.Button(description='Berikutnya')
view_output = widgets.Output()
status_output = widgets.Output()

def current_index():
    return int(pilot_queue.index[candidate_selector.value - 1])

def show_current(*_):
    idx = current_index()
    row = annotations.loc[idx]
    tracking, events = load_cached_match(row['match_id'])
    context = compute_annotation_context(row, tracking, events, CONFIG)
    with view_output:
        clear_output(wait=True)
        fig = plot_annotation_context(context)
        display(fig)
        plt.close(fig)
        display(context['summary'].to_frame('value'))
        existing = row['is_gegenpressing']
        print(f"Tersimpan: {existing if pd.notna(existing) else 'belum'}")
    notes.value = '' if pd.isna(row['annotation_notes']) else str(row['annotation_notes'])
    if pd.notna(row['annotation_confidence']):
        confidence.value = int(row['annotation_confidence'])
    with status_output:
        clear_output(wait=True)
        print('Pilot progress:')
        display(annotation_progress(
            annotations[annotations['pilot_selected']]
        ).to_frame('count'))
        print('Label ragu wajib disertai catatan singkat.')

def save_label(label):
    if label == -1 and not notes.value.strip():
        with status_output:
            clear_output(wait=True)
            print('Tambahkan alasan singkat sebelum menyimpan label Ragu.')
        return
    idx = current_index()
    record_annotation(
        annotations, idx, label, confidence.value, notes.value,
        ANNOTATOR, ANNOTATION_PATH,
    )
    if candidate_selector.value < candidate_selector.max:
        candidate_selector.value += 1
    else:
        show_current()

positive_button.on_click(lambda _: save_label(1))
negative_button.on_click(lambda _: save_label(0))
uncertain_button.on_click(lambda _: save_label(-1))
previous_button.on_click(lambda _: setattr(
    candidate_selector, 'value', max(candidate_selector.min, candidate_selector.value - 1)
))
next_button.on_click(lambda _: setattr(
    candidate_selector, 'value', min(candidate_selector.max, candidate_selector.value + 1)
))
candidate_selector.observe(show_current, names='value')

display(candidate_selector)
display(confidence, notes)
display(widgets.HBox([previous_button, negative_button, uncertain_button, positive_button, next_button]))
display(status_output, view_output)
show_current()

IntSlider(value=1, continuous_update=False, description='Pilot', layout=Layout(width='70%'), max=28, min=1)

ToggleButtons(description='Confidence', index=1, options=(('Rendah', 1), ('Sedang', 2), ('Tinggi', 3)), value=…

Textarea(value='', description='Catatan', layout=Layout(height='70px', width='85%'), placeholder='Alasan keput…

Output()

Output()

## Audit setelah anotasi

Jalankan cell berikut secara berkala. Hubungan dengan quick regain baru dibuka pada tahap audit, setelah keputusan perilaku dibuat.

In [5]:
saved = pd.read_csv(ANNOTATION_PATH)
pilot_mask = saved['pilot_selected'].astype(str).str.lower().eq('true')
pilot_saved = saved[pilot_mask].copy()
print('Pilot progress:')
display(annotation_progress(pilot_saved).to_frame('count'))
print('Full queue progress:')
display(annotation_progress(saved).to_frame('count'))

completed = pilot_saved[pilot_saved['is_gegenpressing'].notna()].copy()
if len(completed):
    completed['is_gegenpressing'] = completed['is_gegenpressing'].astype(int)
    display(pd.crosstab(
        completed['is_gegenpressing'],
        completed['quick_regain_success'],
        margins=True,
    ))
    display(pd.crosstab(
        completed['match_id'],
        completed['is_gegenpressing'],
        margins=True,
    ))
    uncertain_without_notes = completed[
        completed['is_gegenpressing'].eq(-1)
        & completed['annotation_notes'].fillna('').str.strip().eq('')
    ]
    print('Uncertain labels without notes:', len(uncertain_without_notes))
    if len(uncertain_without_notes):
        display(uncertain_without_notes[[
            'pilot_order', 'candidate_id', 'match_id',
            'annotation_confidence',
        ]])
else:
    print('Belum ada label manual yang tersimpan.')

Pilot progress:


,count
total_queue,28
completed,28
remaining,0
gegenpressing,21
non_gegenpressing,5
uncertain,2


Full queue progress:


,count
total_queue,264
completed,28
remaining,236
gegenpressing,21
non_gegenpressing,5
uncertain,2


quick_regain_success,0,1,All
is_gegenpressing,,,
-1,2,0,2
0,2,3,5
1,10,11,21
All,14,14,28


is_gegenpressing,-1,0,1,All
match_id,,,,
J03WMX,0,1,9,10
J03WN1,0,1,2,3
J03WOH,1,0,2,3
J03WOY,0,1,2,3
J03WPY,0,1,2,3
J03WQQ,0,1,2,3
J03WR9,1,0,2,3
All,2,5,21,28


Uncertain labels without notes: 0


## Quality gate sebelum training

Sebelum label dipakai untuk training:
1. Tinjau ulang semua label `-1` dan confidence rendah.
2. Minta anotator kedua menilai minimal 50-60 sampel tanpa melihat label pertama.
3. Laporkan agreement (persentase agreement dan Cohen's kappa).
4. Bekukan versi manifest dan jangan mengubah label berdasarkan outcome quick regain.
5. Gunakan split berbasis pertandingan, bukan random split episode.